# Consolidação — triagem dos erros observados da base 3

> Códigos: [glossário](../../glossario.md). Método: [mineração sem censo](../docs/02-mineracao-sem-censo.md).

**Contrato:** somente o canal estruturado observado. Não exige censo, sucesso por conteúdo ou detector de silenciosas. Não lê os CSVs canônicos copiados da base 2. Não é a consolidação completa dos dois canais.

A fonte precisa abrir e passar pelo adaptador; erro de leitura interrompe antes de qualquer publicação. Outputs restaurados não certificam fonte atual. O inventário e o manifesto da rodada identificam fonte, código, cobertura e hashes de artefatos.

As regras `classify`, `submecanismo`, `SUB2UNI` e `UNI` são reutilizadas. Uma unidade observável é regra/proxy aplicável, não lição validada. Pendências por evidência não são resíduos. Candidatas observadas exigem ≥3 execuções e ≥2 meses; a decisão de memória × harness continua no gate do pesquisador.

## Capacidades e limites

- Custos: somente `n`, soma conhecida acompanhada de custos ausentes; não orçamento/desperdício/economia.
- Ocorrências: componente estruturada × unidade, calculada antes de selecionar erros classificáveis. Vínculos são candidatos; fronteiras abertas são explícitas. Contagem condicional não é episódio exato nem necessariamente limite inferior.
- Recorrência: execuções identificadas por execução + agente, mês real de início. Não é taxa de risco por ActionStep.
- Silenciosas: **não medidas**, não zero. Suspeitas do produtor permanecem fora da taxonomia.
- Localização: `step_pos` original e `step_ref`; `idx` desconhecido não é preenchido por enumeração.
- Saída: subpasta privada da rodada em resultados, nunca os CSVs legados na raiz.

Execute as células deste notebook somente após restaurar/conferir a fonte. O modo Python controlado dos testes usa fixtures sintéticas e não é evidência de uma execução Jupyter na base real.

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
from base_pipeline import carregar_mineracao_observada
from mineracao_observada import CANDIDATO, gravar, grafico_unidades, grafico_papeis, inventariar

A = carregar_mineracao_observada()
TC = A.tabelas['triagem_visivel_observada']
TCS = A.tabelas['triagem_sensibilidade']
print('Contrato:', A.manifesto['contrato'], '· rodada:', A.manifesto['rodada_id'][:16])
print(A.resumo)
print('Silenciosas não medidas; taxas globais e sucesso semântico bloqueados.')

## 14.1 · Cobertura antes da triagem

Todos os erros estruturados são conservados. Custos e contagens das pendências continuam no inventário, mesmo sem unidade. Silenciosas não medidas não participam de nenhuma união artificial.

In [ ]:
display(A.tabelas['cobertura_sintomas'])
display(A.tabelas['cobertura_mecanismos'])
assert A.tabelas['cobertura_mecanismos']['erros'].sum() == len(A.EU)
print('Canais: erros estruturados triados; suspeitas separadas; silenciosas não medidas.')

## 14.2 · Triagem visível observada

Régua: ≥3 execuções e ≥2 meses, sensibilidade ≥5/3. Plataforma não vira candidata; resíduo vai para revisão; críticos vão para investigação. Menor recorrência significa **recorrência não demonstrada neste subconjunto**, não ausência do padrão. Destinos das bases anteriores não são herdados.

Percentual de custo, quando disponível, usa todos os erros estruturados como denominador; não o orçamento da população. A triagem não valida o conteúdo de uma memória.

In [ ]:
cand = TC[TC['decisao'].eq(CANDIDATO)]
firmes = set(TCS.loc[TCS['decisao'].eq(CANDIDATO), 'unidade'])
limitrofes = sorted(set(cand['unidade']) - firmes)
print(f'{len(TC)} unidades observáveis · {len(cand)} candidatas observadas')
print('Distribuição:', TC['decisao'].value_counts().to_dict())
print('Limítrofes na régua ≥5/3:', limitrofes)
display(TC)

## 14.3 · Continuidade e sensibilidade

Não há comparação consolidada versus visível enquanto o canal silencioso estiver não medido. Mostramos componentes com limites abertos/fechados na definição da extração. Nem fechamento nem próxima ação sem sinal provam mesma chamada, mesma causa ou recuperação.

In [ ]:
display(TC[['unidade', 'ocorrencias_condicionais', 'ocorrencias_limites_fechados', 'erros_limites_abertos']])
episodes = A.mecanismos.episodios.episodios
if len(episodes):
    display(episodes.groupby(['definicao', 'inicio', 'fim'], dropna=False).size().reset_index(name='componentes'))
print('Contagens entre definições se sobrepõem; não somar.')

## 14.4 · Unidade observada por papel e mês

O gesto de retorno colado é contado somente onde a regra existente tem evidência suficiente. O catálogo é hipótese de lição; não transplantamos a diretiva nem o destino decidido em outra base.

In [ ]:
g = A.EU[A.EU['unidade'].eq('U_repr_colado')]
display(g.groupby(['role', 'mes']).agg(erros=('step_ref', 'size'), ocorrencias_condicionais=('ocorrencia_observada', 'nunique')))
print('Canal silencioso desta unidade não medido; não usar o total legado da base 2.')

## 14.5 · Fila de complementação

Pendências de insumos não são falhas silenciosas nem resíduos. A tabela conserva os custos dos erros cujo mecanismo não pode ser atribuído. Mensagem limitada/ausente e insuficiência de histórico/prompt/código exigem pedidos distintos.

In [ ]:
coverage = A.tabelas['cobertura_mecanismos']
display(coverage[coverage['situacao_mecanismo'].eq('pendente')])
print('Pedidos por regra, sem exigir censo; não converter pendência em X_ ou não-memória.')

## 14.6 · Custos das unidades observadas

Este gráfico não precisa de orçamento global: soma somente custos conhecidos de `n` dos erros atribuídos. Custos ausentes e pendências ficam nos quadros de cobertura; ranking não é economia evitável nem prioridade definitiva de memória.

In [ ]:
fig = grafico_unidades(A)
plt.show()
plt.close(fig)

## 14.7 · Recorrência dentro de cada papel

Aplicar ≥3 execuções e ≥2 meses dentro do papel. A marca de candidatura significa recorrência demonstrada entre erros atribuídos; as células são contagens, não taxas por ação. Não presumir canal silencioso vazio.

In [ ]:
T4 = A.tabelas['triagem_por_papel']
display(T4)
fig = grafico_papeis(A)
plt.show()
plt.close(fig)

## 14.8 · Global × papel e sensibilidade

Contrastar recorrência observada global e por papel. A régua estrita é ≥5 execuções e ≥3 meses. Meses ausentes na fonte e mecanismos pendentes não demonstram desaparecimento ou correção do padrão.

In [ ]:
global_cand = set(TC.loc[TC['decisao'].eq(CANDIDATO), 'unidade'])
contrast = T4[['role', 'unidade', 'decisao']].copy()
contrast['candidata_global'] = contrast['unidade'].isin(global_cand)
display(contrast)
strict = A.tabelas['triagem_por_papel_sensibilidade'].set_index(['role', 'unidade'])['decisao']
regular = T4.set_index(['role', 'unidade'])['decisao']
changed = (regular.eq(CANDIDATO) & ~strict.reindex(regular.index).eq(CANDIDATO)).sum()
print('Células limítrofes por papel na régua ≥5/3:', int(changed))

## 14.9 · Publicação privada identificada

Gera manifesto, cobertura, triagem, gráficos e genealogia incluindo todos os erros estruturados. Não sobrescreve tabelas canônicas legadas. O manifesto carimba fonte/código e cada arquivo; consumidores rejeitam hashes incompatíveis. Esta publicação técnica não aprova nenhuma memória.

In [ ]:
SAIDA_RODADA = gravar(A)
print('Publicação técnica privada:', SAIDA_RODADA.name)
print('Sem publicação em candidatos_memoria.csv legado; sem aprovação de memória.')